<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 06 — Reporte final**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Resumen ejecutivo

Este proyecto modela las **tasas brutas de mortalidad y natalidad** de los países del mundo (Banco Mundial, 2000–2022) a partir de indicadores socioeconómicos y de estructura etaria. El flujo completo vive en cinco notebooks previos:

| Notebook | Rol |
|---|---|
| `00_setup` | Entorno reproducible (devcontainer, dependencias, PyTorch CPU). |
| `01_eda` | Adquisición vía `wbgapi`, calidad de datos y formulación del problema. |
| `02_preprocesamiento` | Imputación jerárquica, estructura etaria, flags (`gini_imputed`, `is_pandemic`), persistencia. |
| `03_modelado` | Comparación Ridge/RF/XGBoost/LightGBM con `GroupKFold`, robustez de pandemia, interpretación de R². |
| `04_validacion_metodologica` | Verificación de integración + tres problemas metodológicos. |
| `05_optimizacion_avanzada` | GridSearchCV + SHAP + diagnóstico de residuales. |

Este notebook **consolida** los resultados en una *model card*, una tabla final de métricas, las limitaciones y un checklist de reproducibilidad.

> **Autocontenido.** Carga `data/processed/dataset_modelado.parquet` (salida del 02). Requiere haber ejecutado 01 y 02.

## 1. Configuración y carga del dataset

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_validate
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore', category=UserWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_PATH = '../data/processed/dataset_modelado.parquet'
assert os.path.exists(PROCESSED_PATH), (
    "Falta data/processed/dataset_modelado.parquet. Ejecuta los notebooks 01 y 02 (Run All) antes."
)
df = pd.read_parquet(PROCESSED_PATH)
if 'region_' in df.columns:
    df = df.drop(columns='region_')

META_COLS = ['country_code', 'country_name', 'region', 'income_level', 'year']
TARGETS = ['death_rate', 'birth_rate']
FEATURES = [c for c in df.columns if c not in META_COLS + TARGETS]

X = df[FEATURES].values
y_death = df['death_rate'].values
y_birth = df['birth_rate'].values
groups = df['country_code'].values
cv = GroupKFold(n_splits=5)
scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
targets_y = {'death_rate': y_death, 'birth_rate': y_birth}

def construir_modelos(random_state=RANDOM_STATE):
    return {
        'Ridge': Pipeline([('scaler', StandardScaler()),
                           ('reg', Ridge(alpha=1.0, random_state=random_state))]),
        'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=2,
                                              random_state=random_state, n_jobs=-1),
        'XGBoost': xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                    subsample=0.9, colsample_bytree=0.9,
                                    random_state=random_state, n_jobs=-1, verbosity=0),
        'LightGBM': lgb.LGBMRegressor(n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
                                      subsample=0.9, colsample_bytree=0.9,
                                      random_state=random_state, n_jobs=-1, verbose=-1),
    }

print(f"Dataset: {df.shape} | features: {len(FEATURES)} | países: {df['country_code'].nunique()}")

## 2. Tabla final de métricas (validación cruzada por país)

Se reconsolida la comparación de los cuatro modelos para ambos targets bajo `GroupKFold` (5 pliegues, agrupado por país). Es la tabla de referencia del proyecto.

In [ ]:
rows = []
for tn, y in targets_y.items():
    for mn, m in construir_modelos().items():
        sc = cross_validate(m, X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
        rows.append({'target': tn, 'modelo': mn,
                     'RMSE': -sc['test_rmse'].mean(),
                     'MAE': -sc['test_mae'].mean(),
                     'R2': sc['test_r2'].mean()})
tabla = pd.DataFrame(rows).sort_values(['target', 'RMSE']).reset_index(drop=True)
print("Comparación de modelos (GroupKFold por país):\n")
print(tabla.round(3).to_string(index=False))

print("\nMejor modelo por target (menor RMSE):")
mejores = tabla.loc[tabla.groupby('target')['RMSE'].idxmin()]
print(mejores[['target', 'modelo', 'RMSE', 'R2']].round(3).to_string(index=False))

## 3. Señal socioeconómica honesta (sin estructura etaria)

Dado que `pop_0to14_pct` correlaciona ~0.98 con la natalidad (identidad demográfica), la cifra **honesta** de cuánta señal aportan los predictores *socioeconómicos puros* se obtiene excluyendo la estructura etaria.

In [ ]:
FEAT_SE = [c for c in FEATURES if c not in ['pop_0to14_pct', 'pop_65plus_pct']]
X_se = df[FEAT_SE].values
print("XGBoost — R² con vs. sin estructura etaria (GroupKFold):\n")
for tn, y in targets_y.items():
    sc_full = cross_validate(construir_modelos()['XGBoost'], X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
    sc_se = cross_validate(construir_modelos()['XGBoost'], X_se, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"  {tn:11s}  R² con edad = {sc_full['test_r2'].mean():.3f}  |  "
          f"R² solo socioeconómicos = {sc_se['test_r2'].mean():.3f}")

## 4. Model card

**Modelo:** XGBoost (gradient boosting de árboles), un modelo independiente por target.
**Tarea:** regresión supervisada de `death_rate` y `birth_rate` (eventos por 1.000 hab.).
**Datos:** Banco Mundial vía `wbgapi`, 2000–2022, ~200–220 países (agregados regionales excluidos).
**Features:** indicadores económicos (PIB per cápita log, crecimiento, Gini, desempleo), sociales (gasto en salud/educación %PIB, urbanización), estructura etaria (`pop_65plus_pct`, `pop_0to14_pct`), codificaciones (región one-hot, ingreso ordinal), temporales (`year_norm`, `is_pandemic`) y bandera `gini_imputed`.
**Validación:** `GroupKFold` por país (5 pliegues) → mide **generalización a países no observados**.
**Métricas:** ver tabla de la sección 2 (RMSE en unidades del target; R² adimensional).

**Uso previsto:** análisis exploratorio y educativo del vínculo entre desarrollo socioeconómico y dinámica demográfica. **No** es una herramienta de proyección demográfica oficial.

**Consideraciones éticas / de interpretación:**
- El R² de natalidad está inflado por la identidad demográfica con `pop_0to14_pct`; la cifra honesta es la de la sección 3.
- El Gini mide **solo desigualdad de ingresos** y está fuertemente imputado.
- Las tasas objetivo son **estimaciones modeladas** del Banco Mundial, no registro civil crudo.
- No usar para decisiones individuales ni para comparaciones país a país fuera del esquema de validación empleado.

## 5. Limitaciones

- **Ground truth modelado.** Las tasas son estimaciones del Banco Mundial → techo de desempeño y residuales irreducibles (notebook 04, P2).
- **Cobertura/alcance del Gini.** Solo ingresos, ~71% imputado (notebook 04, P1).
- **Ausencia de variables institucionales y sanitarias** (gobernanza, capacidad hospitalaria, conflictos), que explicarían parte de los residuales.
- **Generalización ≠ proyección.** `GroupKFold` evalúa países nuevos, no el futuro; un objetivo de pronóstico exigiría partición temporal.
- **Choque pandémico.** Capturado con `is_pandemic`; el modelo no explica causalmente el exceso de mortalidad de 2020–2021.

## 6. Checklist de reproducibilidad

- [x] **Entorno fijo:** devcontainer (Python 3.13) + `requirements.txt` con versiones acotadas; PyTorch desde índice CPU.
- [x] **Dependencias de datos:** `wbgapi` (API) y `pyarrow` (Parquet) declaradas.
- [x] **Datos regenerables por código:** `data/` no se versiona; se reconstruye ejecutando los notebooks.
- [x] **Semilla:** `RANDOM_STATE = 42` en preprocesamiento y modelado.
- [x] **Validación honesta:** `GroupKFold` por país (sin fuga de datos panel).
- [x] **Pipeline verificado:** celda de integración en el notebook 04.
- [x] **Trazabilidad de cambios:** `AUDITORIA.md` + `CHANGELOG.md` (ramas, PRs y merges).

**Orden de ejecución:** `00 → 01 → 02 → 03 → 04 → 05 → 06`. Tras cambiar dependencias, **Rebuild Container** en Codespaces.

## 7. Conclusiones del proyecto

1. La natalidad es altamente predecible, pero su R² está dominado por una **identidad demográfica**; la señal socioeconómica honesta (sección 3) es más moderada y es la cifra a comunicar.
2. La mortalidad tiene un **techo estructural** más bajo: depende de la composición etaria y de factores idiosincráticos ausentes del set. Los modelos de árboles superan claramente al lineal, confirmando la **no linealidad** (paradoja demográfica en U invertida).
3. Los resultados son **robustos** a la exclusión de los años de pandemia.
4. El proyecto deja un pipeline **reproducible, verificado y trazable**, con las salvedades metodológicas explícitas (alcance del Gini, *ground truth*, outliers por tipo).

**Trabajo futuro:** incorporar variables institucionales/sanitarias, evaluar partición temporal para proyección y profundizar la interpretabilidad SHAP por región e ingreso.